# Guía: crear y distribuir un paquete de Python

---

Vamos a ver cómo  crear un proyecto, organizar el código como paquete, añadir pruebas, construir el paquete e instalarlo en otro proyecto.

Se utilizará `uv` para gestionar el proyecto y el entorno virtual.

Los ficheros necesarios para crear el paquete se encuentran en la carpeta: `files_guia_package`.

## 1. Crear el proyecto

Abre una terminal y sitúate en la carpeta donde quieras guardar el proyecto. No lo crees dentro de otro proyecto que ya tenga un entorno virtual, porque `uv` puede heredarlo.

Cambia `ruta\ejemplo_paquete` por la ruta que quieras utilizar.

```python
    uv init --package ruta\ejemplo_paquete
    cd ruta\ejemplo_paquete
```

## 2. Revisar la estructura inicial

El proyecto debe tener una estructura similar a esta. El código distribuible se coloca dentro de `src/`; los tests quedan fuera del paquete que se instalará.



<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 2rem;">

  <div>
       <p><strong>Ejemplo 1</strong></p>
    <p><div style="width:90%; text-align: center;">
     </br>
    <img src="images\Picture1.png" style="width: 100%;"/>
</br> </br>   
</div></p>
  </div>

  <div>
       <p><strong>Ejemplo 2 con tests (con uv)</strong></p>
    <p><div style="width: 90%; text-align: center;">
     </br>
    <img src="images\Picture2.png" style="width: 100%;"/>
</br> </br>   
</div></p>
  </div>

</div>

Los sistemas de empaquetado modernos detectan automáticamente el directorio `src/` y empaquetan únicamente lo que esté dentro de él. De esta forma, la carpeta `tests` no se incluirá dentro del `.wheel` generado.



## 3. Configurar `pyproject.toml`

Comprueba que el archivo contiene el nombre del proyecto, la versión y el sistema de construcción. En este ejemplo se utiliza `uv_build`.

```python
    [project]
    name = "ejemplo-paquete"
    version = "0.1.0"
    description = "Un paquete de ejemplo"
    readme = "README.md"
    requires-python = ">=3.11"
    dependencies = []

    [build-system]
    requires = ["uv_build>=0.10.2,<0.11.0"]
    build-backend = "uv_build"

```

## 4. Instalar herramientas de desarrollo

Añade `pytest`, `coverage` y el plugin de cobertura como dependencias de desarrollo.
Se creará un nuevo entorno virtual.

```python

    uv add --dev pytest coverage pytest-cov pytest-datafiles
```

<div style="width: 90%; text-align: center;">
     </br>
    <img src="images\Picture3.png" style="width: 30%;"/>
</br> </br>   
</div>

## 5. Crear los subpaquetes

Dentro de `src/ejemplo_paquete/`, crea los paquetes `rationals` y `polynomials`. Cada uno debe tener un archivo `__init__.py`. Para crear los subpaquetes puedes usar el menú de Pycharm (New -> Python Package).

<div style="width: 90%; text-align: center;">
     </br>
    <img src="images\Picture4.png" style="width: 30%;"/>
</br> </br>   
</div>

El paquete `polynomials` podrá importar clases del paquete hermano `rationals` mediante importaciones relativas.

## 6. Añadir módulos e interfaz pública

### Módulos

Crea `rational.py`, `mono.py` y `poly.py` con las implementaciones proporcionadas para el ejercicio.

* `mono.py`: Este módulo usa la clase `Rational` definida en el módulo rational.py. Fíjate que la primera línea en mono.py incluye la importación:

```python
       from ..rationals import Rational
```
En este módulo se define la clase `Monomial`.

* `poly.py`: Este módulo usa las clases `Monomial` y `Rational`, por lo que hace las importaciones:

```python
       from .mono import Monomial
       from ..rationals import Rational
```

En este módulo se define la clase `Polynomial`.

<div style="width: 90%; text-align: center;">
     </br>
    <img src="images\Picture5.png" style="width: 30%;"/>
</br> </br>   
</div>

### Interfaz pública

Después define la interfaz pública de cada subpaquete mediante `__all__`. Las importaciones relativas evitan depender de la carpeta desde la que se ejecute el programa.

```python
    # src/ejemplo_paquete/rationals/__init__.py
    from .rational import Rational

    __all__ = ["Rational"]    # interfaz

    # src/ejemplo_paquete/polynomials/__init__.py
    from .mono import Monomial
    from .poly import Polynomial

    __all__ = ["Monomial", "Polynomial"]
```

## 7. Probar un módulo directamente

### Opción 1

Puedes incluir un bloque `if __name__ == "__main__":` para hacer una prueba manual y ejecutar desde la terminal el módulo con `-m` desde la raíz del proyecto.

```python
    uv run -m ejemplo_paquete.rationals.rational
```

El módulo `rational.py` puede contener:

```Python
        if __name__ == "__main__":
            r1 = Rational(2, 4)
            print(r1)
```

`if __name__ == "__main__":`  evalúa si un archivo de Python se está ejecutando directamente desde la terminal o si se ha importado desde otro script. 
* Si el archivo se lanza de forma directa, la variable especial __name__ adopta el valor "__main__" y el código dentro del bloque se ejecuta; 
* si en cambio se importa como módulo en otro proyecto, dicha variable toma el nombre del archivo y el bloque se ignora, evitando así la ejecución accidental de pruebas, scripts de consola o código secundario.

### Opción 2

Otra forma de probar los módulos es crear un módulo `run.py` junto al archivo `.toml` y ejecutarlo con :

```python
    uv run run.py   
```



<div style="width: 90%; text-align: center;">
     </br>
    <img src="images\Picture6.png" style="width: 30%;"/>
</br> </br>   
</div>


Este fichero solo lo queremos para hacer pruebas y depurar, no queremos incluirlo en el paquete. Si usamos `uv`, no neceses necesario configurar exclusiones en el .toml, ya que los empaquetadores solo incluyen por defecto lo que reside dentro de src/ejemplo_paquete/.

## 8. Crear y ejecutar tests

Copia los tests en la carpeta `tests/`, por ejemplo `test_rationals.py`, `test_monomials.py` y `test_polynomials.py`. Ejecuta todos los tests, incluyendo los doctests, y genera el informe HTML de cobertura.

```python
uv run python -m pytest --doctest-modules --cov=. --cov-report=html
# Abre htmlcov/index.html para consultar el informe
```

<div style="width: 90%; text-align: center;">
     </br>
    <img src="images\Picture7.png" style="width: 50%;"/>
</br> </br>   
</div>



También puedes ejercutar los tests de forma individual (botón derecho del ratón sobre el archivo -> More Run/Debug -> Run Python test in test with coverage).

## 9. Construir las distribuciones

Desde la raíz del proyecto, donde está `pyproject.toml`, construye el paquete. Se generarán una distribución fuente (`.tar.gz`) y un archivo wheel (`.whl`) dentro de `dist/`.

```python
        uv build
        # Para generar solamente el wheel:
        uv build --wheel
```

<div style="width: 90%; text-align: center;">
     </br>
    <img src="images\Picture8.png" style="width: 50%;"/>
</br> </br>   
</div>

## 10. Instalar y probar el wheel

En otro proyecto, instala el archivo `.whl` generado. Sustituye la ruta y el nombre por los que existan en tu equipo.

```python
uv add ruta\ejemplo_paquete\dist\ejemplo_paquete-0.1.0-py3-none-any.whl
uv run python -c "from ejemplo_paquete.rationals import Rational; print(Rational(2, 3))"
```

En el nuevo proyecto puedes crear un nuevo módulo para probar. Por ejemplo:

```python
# módulo main.py

from ejemplo_paquete import rationals

def main():
    print("Hello from probar-paquete!")
    a = rationals.Rational(2, 3)                # __all__ = ["Rational"]
    r = rationals.rational.gcd(4, 6)            # __all__ = ["Rational"]
    print(a,r)

if __name__ == "__main__":
    main()
```

Si necesitamos cambiar el código fuente del paquete sin cambiar la versión en el toml, tenemos que volver a instalarlo y forzar la reinstalación con `--reinstall`. Lo mejor es usar uv con pip:

 ```python 
 uv pip install ruta\ejemplo_paquete\dist\ejemplo_paquete-0.1.0-py3-none-any.whl --reinstall 
 ```
Esto solo lo mete a la fuerza en tu entorno virtual sin actualizar el archivo de bloqueo `uv.lock`.
